In [ ]:
#Requires BioMedParse key from huggingface https://huggingface.co/microsoft/BiomedParse
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
"""
Inference for BioMedParse
Must recieve access from Microsoft through huggingface to download the model
Change Text Prompt in Cell 4
"""

# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q hydra-core huggingface_hub scikit-image nibabel pandas scikit-learn scipy
!pip install -q timm transformers opencv-python Pillow

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

!pip install -q einops fvcore iopath



# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Download dataset from Kaggle

import os

# Install kaggle
!pip install -q kaggle

# Create kaggle directory
!mkdir -p ~/.kaggle

# Check if kaggle.json already exists
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("\n Upload kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()

    # Move kaggle.json to correct location
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
DATASET_PATH = "/content/dataset/lgg-mri-segmentation/kaggle_3m"
if not os.path.exists(DATASET_PATH):
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q lgg-mri-segmentation.zip -d /content/dataset

    # Verify extraction
    if os.path.exists(DATASET_PATH):
        print("Dataset downloaded ")
    else:
        # Check alternate locations
        import glob
        data_csv_files = glob.glob("/content/dataset/**/data.csv", recursive=True)
        if data_csv_files:
            DATASET_PATH = os.path.dirname(data_csv_files[0])
            print(f"Dataset found at: {DATASET_PATH}")
else:
    print("Dataset already exists")

print(f"\nDataset location: {DATASET_PATH}")

# Clone BiomedParse repo
import os
if not os.path.exists('/content/BiomedParse'):
    !git clone https://github.com/microsoft/BiomedParse.git

# Add to path
import sys
sys.path.insert(0, '/content/BiomedParse/src')
print("BiomedParse added to path")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.3 MB/s eta 0:00:00
Mounted at /content/drive

 Upload kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation
License(s): CC-BY-NC-SA-4.0
100% 714M/714M [00:05<00:00, 143MB/s]

Dataset downloaded 

Dataset location: /content/dataset/lgg-mri-segmentation/kaggle_3m
Cloning into 'BiomedParse'...
remote: Enumerating objects: 1865, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 1865 (delta 110), reused 96 (delta 95), pack-reused 1703 (from 2)
Receiving objects: 100% (1865/1865), 674.43 MiB | 26.95 MiB/s, done.
Resolving deltas: 100% (598/598), done.
Updating files: 100% (533/533), done.
Filtering content: 100% (46/46), 69.69 MiB | 26.45 MiB/s, done.
BiomedParse added to path


In [ ]:
#Load Model

import torch
import numpy as np
import nibabel as nib
import pandas as pd
from PIL import Image
from glob import glob
import torch.nn.functional as F
import hydra
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, jaccard_score
import random
import torch.nn as nn
from contextlib import contextmanager
import copy

#Fix - Change to correct Directory
import os
os.chdir('/content/BiomedParse')
print(f"  Working directory: {os.getcwd()}")

# Verify configs exist
config_path = os.path.abspath('configs/model')
if os.path.exists(config_path):
    print(f"  Configs at: {config_path}")
else:
    print(f"  Configs directory not found at {config_path}!")
    raise FileNotFoundError(f"configs/model directory not found at {config_path}")

# Import BiomedParse utilities
import sys
if '/content/BiomedParse/src' not in sys.path:
    sys.path.insert(0, '/content/BiomedParse/src')

from utils import process_input, process_output
from inference import postprocess, merge_multiclass_masks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Initialize Hydra with ABSOLUTE path
GlobalHydra.instance().clear()
hydra.initialize_config_dir(
    config_dir=config_path,
    job_name="batch_prediction",
    version_base=None
)
cfg = compose(config_name="biomedparse_3D")
model = hydra.utils.instantiate(cfg, _convert_="object")

# Download and load checkpoint
checkpoint_path = hf_hub_download(
    repo_id="microsoft/BiomedParse",
    filename="biomedparse_v2.ckpt"
)
model.load_pretrained(checkpoint_path)
model = model.to(device).eval()


  Working directory: /content/BiomedParse
  Configs at: /content/BiomedParse/configs/model
  Device: cuda


/usr/local/lib/python3.12/dist-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'biomedparse_3D': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

biomedparse_v2.ckpt:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

Checkpoint loaded successfully!


In [ ]:
# =============================================================================
# Soft Prompting + Anchor Prompts (SEEMLanguageEncoder / GitHub BiomedParse)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import contextmanager

# --- 1) Grab language encoder ---
lang_enc = model.sem_seg_head.predictor.language_encoder
TEXT_EMBED_DIM = lang_enc.encoder_transformer.token_embedding.embedding_dim  # 512
print('[SoftPrompt] lang encoder:', type(lang_enc))
print('[SoftPrompt] TEXT_EMBED_DIM:', TEXT_EMBED_DIM)

# --- 2) Soft prompt module ---
class SoftPrompt(nn.Module):
    def __init__(self, num_tokens: int, embed_dim: int):
        super().__init__()
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim
        self.prompt_embeddings = nn.Parameter(torch.randn(num_tokens, embed_dim) * 0.02)

    def forward(self, batch_size: int) -> torch.Tensor:
        return self.prompt_embeddings.unsqueeze(0).expand(batch_size, -1, -1)  # (B,T,D)

# --- 3) Patch get_text_token_embeddings to prepend soft tokens ---
@contextmanager
def patch_get_text_token_embeddings(lang_enc, soft_prompt):
    orig = lang_enc.get_text_token_embeddings

    def patched(txts, name='default', token=False, norm=False, *args, **kwargs):
        out = orig(txts, name=name, token=token, norm=norm, *args, **kwargs)
        if not isinstance(out, dict) or 'token_emb' not in out or 'tokens' not in out:
            raise RuntimeError(
                f'Unexpected output from get_text_token_embeddings: '
                f'{type(out)} keys={getattr(out, "keys", lambda: [])()}'
            )
        tok    = out['token_emb']   # (B,L,D)
        tokens = out['tokens']      # dict with input_ids + attention_mask
        if not torch.is_tensor(tok) or tok.dim() != 3:
            return out
        B, L, D = tok.shape
        T = soft_prompt.num_tokens
        soft = soft_prompt(B).to(tok.dtype).to(tok.device)  # (B,T,D)
        tok2 = torch.cat([soft, tok], dim=1)                # (B,T+L,D)
        attn = tokens.get('attention_mask', None)
        if attn is not None and torch.is_tensor(attn):
            soft_mask = torch.ones(B, T, dtype=attn.dtype, device=attn.device)
            attn2 = torch.cat([soft_mask, attn], dim=1)
        else:
            attn2 = None
        ids = tokens.get('input_ids', None)
        if ids is not None and torch.is_tensor(ids):
            pad_id = getattr(lang_enc.tokenizer, 'pad_token_id', 0)
            if pad_id is None:
                pad_id = 0
            soft_ids = torch.full((B, T), int(pad_id), dtype=ids.dtype, device=ids.device)
            ids2 = torch.cat([soft_ids, ids], dim=1)
        else:
            ids2 = None
        tokens2 = dict(tokens)
        if attn2 is not None: tokens2['attention_mask'] = attn2
        if ids2  is not None: tokens2['input_ids']      = ids2
        out2 = dict(out)
        out2['token_emb'] = tok2
        out2['tokens']    = tokens2
        return out2

    lang_enc.get_text_token_embeddings = patched
    try:
        yield
    finally:
        lang_enc.get_text_token_embeddings = orig

# --- 4) Anchor prompt manager ---
class AnchorPromptManager:
    def __init__(self, anchor_phrases, lang_enc, device):
        self.device = device
        self.anchor_phrases = anchor_phrases
        self.anchor_embeddings = self._encode_anchors(anchor_phrases, lang_enc)
        print(f'[AnchorPromptManager] {len(anchor_phrases)} anchors, dim={self.anchor_embeddings.shape[-1]}')

    @torch.no_grad()
    def _encode_anchors(self, phrases, lang_enc):
        lang_enc.eval()
        out = lang_enc.get_text_token_embeddings(phrases, name='default', token=False, norm=False)
        if not isinstance(out, dict) or 'class_emb' not in out:
            raise RuntimeError(f'Unexpected output keys: {getattr(out, "keys", lambda: [])()}')
        return out['class_emb'].to(self.device)

    def init_soft_prompt(self, soft_prompt, lang_enc, top_k_vocab=50, jitter_std=0.0):
        # Deterministic vocabulary-seeded warm init.
        # jitter_std=0.0 by default: fully reproducible, no randomness.
        with torch.no_grad():
            vocab_emb = lang_enc.encoder_transformer.token_embedding.weight  # (V, D)
            vocab_n   = F.normalize(vocab_emb, dim=-1)
            tokenizer = lang_enc.tokenizer
            n_anchors = self.anchor_embeddings.shape[0]

            import nltk
            nltk.download('words', quiet=True)
            from nltk.corpus import words as nltk_words
            english_words = set(w.lower() for w in nltk_words.words()
                                if len(w) >= 4 and w.isalpha())

            valid_ids = []
            for tok_id in range(vocab_emb.shape[0]):
                word = tokenizer.decode([tok_id]).strip().lower()
                if word in english_words:
                    valid_ids.append(tok_id)

            valid_ids   = torch.tensor(valid_ids, device=vocab_emb.device)
            valid_emb_n = vocab_n[valid_ids]
            print(f'  [init] {len(valid_ids)} real English word tokens in vocabulary')

            for i in range(soft_prompt.num_tokens):
                anchor   = self.anchor_embeddings[i % n_anchors]
                anchor_n = F.normalize(anchor.unsqueeze(0), dim=-1)
                sims       = (anchor_n @ valid_emb_n.t()).squeeze(0)
                # Deterministic: always take top-1, no random sampling
                top_idx    = sims.topk(min(top_k_vocab, len(valid_ids))).indices
                chosen_idx = top_idx[0]
                chosen_id  = valid_ids[chosen_idx]
                start_emb  = vocab_emb[chosen_id]
                soft_prompt.prompt_embeddings[i] = (
                    start_emb + torch.randn_like(start_emb) * jitter_std
                )
                word = tokenizer.decode([chosen_id.item()]).strip()
                print(f'  Token {i:2d} (anchor: "{self.anchor_phrases[i % n_anchors]}")'
                      f' -> init word: "{word}"')

        print('[AnchorPromptManager] Vocabulary-seeded warm-init complete')

    def anchor_loss(self, soft_prompt, weight=0.1):
        tokens_n  = F.normalize(soft_prompt.prompt_embeddings, dim=-1)
        anchors_n = F.normalize(self.anchor_embeddings, dim=-1)
        sim       = tokens_n @ anchors_n.t()
        best_sim, _ = sim.max(dim=-1)
        return weight * (1.0 - best_sim).mean()

    def diversity_loss(self, soft_prompt, weight=0.1):
        tokens_n   = F.normalize(soft_prompt.prompt_embeddings, dim=-1)
        sim_matrix = tokens_n @ tokens_n.t()
        mask     = 1 - torch.eye(soft_prompt.num_tokens, device=sim_matrix.device)
        off_diag = sim_matrix * mask
        return weight * off_diag.mean()

# --- 5) Config ---
USE_SOFT_PROMPT = False
NUM_SOFT_TOKENS = 16
ANCHOR_WEIGHT   = 0.1

ANCHOR_PHRASES = [
    'Brain Tumor in MRI',
]

soft_prompt    = SoftPrompt(NUM_SOFT_TOKENS, TEXT_EMBED_DIM).to(device)
anchor_manager = AnchorPromptManager(ANCHOR_PHRASES, lang_enc, device)
# jitter_std=0.0 -> fully deterministic seed, no randomness in initialisation
anchor_manager.init_soft_prompt(soft_prompt, lang_enc, top_k_vocab=50, jitter_std=0.0)

for p in model.parameters():
    p.requires_grad = False
for p in soft_prompt.parameters():
    p.requires_grad = True

print(f'[SoftPrompt] Enabled={USE_SOFT_PROMPT} tokens={NUM_SOFT_TOKENS} '
      f'dim={TEXT_EMBED_DIM} trainable={NUM_SOFT_TOKENS * TEXT_EMBED_DIM}')


[SoftPrompt] lang encoder: <class 'src.model.transformer_decoder.language_encoders.seem_language_encoder.SEEMLanguageEncoder'>
[SoftPrompt] TEXT_EMBED_DIM: 512
[AnchorPromptManager] 1 anchors, dim=512
  [init] 18362 real English word tokens in vocabulary
  Token  0 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  1 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  2 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  3 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  4 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  5 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  6 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  7 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  8 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token  9 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token 10 (anchor: "Brain Tumor in MRI") -> init word: "have"
  Token 11 (anchor: "Brain Tumor in MRI") -> init wo

In [ ]:
bce_logits = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-6):
    # logits/targets: (B,1,H,W)
    probs = torch.sigmoid(logits).view(logits.size(0), -1)
    t = targets.view(targets.size(0), -1)
    inter = (probs * t).sum(1)
    denom = probs.sum(1) + t.sum(1)
    return 1 - ((2*inter + eps) / (denom + eps)).mean()

def list_slice_pairs(patient_dir):
    imgs = sorted([f for f in glob(os.path.join(patient_dir, "*.tif"))
                   if f[-5].isdigit() and "_mask" not in f])
    return [(p, os.path.splitext(p)[0] + "_mask.tif")
            for p in imgs if os.path.exists(os.path.splitext(p)[0] + "_mask.tif")]

def load_slice(img_path, mask_path):
    img  = np.array(Image.open(img_path).convert("L"))                         # (H,W)
    mask = (np.array(Image.open(mask_path).convert("L")) > 0).astype(np.uint8) # (H,W)
    return img, mask
@torch.no_grad()
def evaluate_iou(val_pairs, text_prompt, num_samples=50):
    """
    Estimate mean IoU on a random sample of (image, mask) pairs.
    Matches the main evaluation logic: both empty = 1.0, one empty = 0.0,
    both non-empty = jaccard_score.
    """
    soft_prompt.eval()
    samples = random.sample(val_pairs, min(num_samples, len(val_pairs)))
    ious = []

    for img_path, mask_path in samples:
        img_np, mask_np = load_slice(img_path, mask_path)

        logits = forward_one_slice_logits(img_np, text_prompt)
        pred = (torch.sigmoid(logits) > 0.5).cpu().numpy().squeeze()  # (H,W)

        # Resize pred to match mask if needed
        if pred.shape != mask_np.shape:
            from scipy.ndimage import zoom
            scale = [mask_np.shape[0] / pred.shape[0],
                     mask_np.shape[1] / pred.shape[1]]
            pred = zoom(pred.astype(np.float32), scale, order=0).astype(np.uint8)

        gt_sum   = mask_np.sum()
        pred_sum = pred.sum()

        if gt_sum == 0 and pred_sum == 0:
            ious.append(1.0)
        elif gt_sum == 0 and pred_sum > 0:
            ious.append(0.0)
        elif gt_sum > 0 and pred_sum == 0:
            ious.append(0.0)
        else:
            ious.append(jaccard_score(mask_np.flatten(), pred.flatten(), zero_division=0))

    soft_prompt.train()
    return float(np.mean(ious)) if ious else 0.0

In [ ]:
# =============================================================================
# Medical Verbalization: Slot-Structured Medical Verbalization (SSMV)
# =============================================================================
# Instead of a flat nearest-neighbour lookup over the full vocabulary, SSMV
# maps each soft token into one of four clinically-grounded semantic slots:
#
#   Slot 0 - Lesion type        e.g. 'glioma', 'neoplasm', 'mass'
#   Slot 1 - Anatomical region  e.g. 'cerebral', 'temporal', 'frontal'
#   Slot 2 - Imaging modality   e.g. 'FLAIR', 'gadolinium', 'T1-weighted'
#   Slot 3 - Morphological cue  e.g. 'hyperintense', 'ring-enhancing'
#
# Tokens are assigned to slots cyclically: token i -> slot i % 4.
# This gives a structured, interpretable prompt that reads as a real clinical
# description and adds methodological novelty vs. raw cosine lookup.
#
# Paper framing:
#   'We propose Slot-Structured Medical Verbalization (SSMV), which
#    decomposes the learned soft prompt into four clinical axes -- lesion
#    semantics, neuroanatomy, MRI protocol, and imaging morphology -- and
#    reconstructs a natural-language clinical descriptor by greedily
#    matching each token to its nearest term within its assigned slot.'
# =============================================================================

vocab_embeddings = lang_enc.encoder_transformer.token_embedding.weight  # (V, D)

MEDICAL_SLOTS = {
    'lesion': [
        'glioma', 'glioblastoma', 'astrocytoma', 'oligodendroglioma',
        'meningioma', 'neoplasm', 'tumor', 'mass', 'lesion',
        'carcinoma', 'lymphoma', 'metastasis', 'adenoma',
    ],
    'anatomy': [
        'cerebral', 'cortical', 'subcortical', 'temporal', 'frontal',
        'parietal', 'occipital', 'thalamic', 'basal', 'ganglia',
        'periventricular', 'corpus', 'callosum', 'brainstem', 'cerebellar',
    ],
    'modality': [
        'T1-weighted', 'T2-weighted', 'FLAIR', 'gadolinium', 'contrast-enhanced',
        'diffusion', 'perfusion', 'spectroscopy', 'axial', 'coronal',
        'sagittal', 'post-contrast', 'MRI', 'magnetic',
    ],
    'morphology': [
        'hyperintense', 'hypointense', 'heterogeneous', 'homogeneous',
        'ring-enhancing', 'infiltrative', 'circumscribed', 'edematous',
        'necrotic', 'cystic', 'solid', 'calcified', 'vascular',
        'mass-effect',
    ],
}
SLOT_ORDER = ['lesion', 'anatomy', 'modality', 'morphology']

@torch.no_grad()
def _embed_term_bank(lang_enc, device):
    tokenizer = lang_enc.tokenizer
    vocab_emb = lang_enc.encoder_transformer.token_embedding.weight
    slot_data = {}
    for slot, terms in MEDICAL_SLOTS.items():
        term_vecs, valid_terms = [], []
        for term in terms:
            ids = tokenizer.encode(term, add_special_tokens=False)
            if not ids:
                continue
            vecs = vocab_emb[torch.tensor(ids, device=device)]
            mean_vec = vecs.mean(0)
            term_vecs.append(F.normalize(mean_vec, dim=-1))
            valid_terms.append(term)
        if term_vecs:
            slot_data[slot] = {
                'terms': valid_terms,
                'embs':  torch.stack(term_vecs, dim=0),
            }
    return slot_data

_term_bank = _embed_term_bank(lang_enc, device)
print('[SSMV] Medical term bank built:')
for slot, data in _term_bank.items():
    print(f'  {slot:12s}: {len(data["terms"])} terms')

@torch.no_grad()
def verbalize_soft_prompt_medical(soft_prompt, lang_enc, top_k=3):
    soft_n = F.normalize(soft_prompt.prompt_embeddings.detach(), dim=-1)
    T = soft_prompt.num_tokens
    slot_results = []
    prompt_parts_by_slot = {s: [] for s in SLOT_ORDER}

    for i in range(T):
        slot_name = SLOT_ORDER[i % len(SLOT_ORDER)]
        data = _term_bank.get(slot_name)
        if data is None:
            continue
        tok_vec = soft_n[i]
        sims    = (tok_vec.unsqueeze(0) @ data['embs'].t()).squeeze(0)
        k       = min(top_k, len(data['terms']))
        scores, idxs = sims.topk(k)
        matches = [(data['terms'][j], round(scores[idx].item(), 4))
                   for idx, j in enumerate(idxs.tolist())]
        slot_results.append({'slot': slot_name, 'token_idx': i, 'top_matches': matches})
        prompt_parts_by_slot[slot_name].append(matches[0][0])

    def _pick(slot):
        words = prompt_parts_by_slot.get(slot, [])
        return words[0] if words else ''

    lesion   = _pick('lesion')
    anatomy  = _pick('anatomy')
    modality = _pick('modality')
    morph    = _pick('morphology')

    verbal_prompt = (
        f'{anatomy} {lesion} on {modality} MRI showing {morph} signal'
    ).strip()

    return slot_results, verbal_prompt

def print_verbalization(soft_prompt, lang_enc, top_k=3):
    slot_results, verbal_prompt = verbalize_soft_prompt_medical(
        soft_prompt, lang_enc, top_k=top_k
    )
    print('\n[SSMV] Slot-Structured Medical Verbalization:')
    print(f'  {"Token":<8} {"Slot":<14} Top matches')
    print(f'  {"-"*65}')
    for res in slot_results:
        cands = ', '.join(f'"{w}" ({s:.3f})' for w, s in res['top_matches'])
        print(f'  Token {res["token_idx"]:2d}  [{res["slot"]:<12}]  {cands}')
    print(f'\n  -> Clinical prompt: "{verbal_prompt}"')
    return verbal_prompt


[SSMV] Medical term bank built:
  lesion      : 13 terms
  anatomy     : 15 terms
  modality    : 14 terms
  morphology  : 14 terms


In [ ]:
def forward_one_slice_logits(img_np, text_prompt):
    """
    Runs a single 2D slice through the 3D pipeline (depth-1 volume).
    Returns logits (1,1,512,512) for BCE/Dice loss.
    Called during TRAINING — no torch.no_grad() so gradients reach soft_prompt.
    """
    vol = img_np[None, ...]  # (D=1, H, W)

    imgs, pad_width, padded_size, valid_axis = process_input(vol, 512)
    imgs = imgs.to(device).int()

    input_tensor = {"image": imgs.unsqueeze(0), "text": [text_prompt]}
    # Note: do NOT add a "no_grad" key — BiomedParse ignores it and it's misleading

    if USE_SOFT_PROMPT:
        with patch_get_text_token_embeddings(lang_enc, soft_prompt):
            out = model(input_tensor, mode="eval", slice_batch_size=1)
    else:
        out = model(input_tensor, mode="eval", slice_batch_size=1)

    gm = out["predictions"]["pred_gmasks"]

    # Robust to shape variations: (B,N,D,h,w) | (B,D,h,w) | (B,N,h,w)
    if gm.dim() == 5:
        gm = gm[:, :, 0, :, :]                      # (B,N,h,w)
        gm = gm.max(dim=1, keepdim=True).values      # (B,1,h,w)
    elif gm.dim() == 4:
        if gm.size(1) != 1:
            gm = gm.max(dim=1, keepdim=True).values  # (B,1,h,w)
    elif gm.dim() == 3:
        gm = gm.unsqueeze(1)
    else:
        raise RuntimeError(f"Unexpected pred_gmasks shape: {tuple(gm.shape)}")

    logits = F.interpolate(gm, size=(512, 512), mode="bicubic",
                           align_corners=False, antialias=True)
    return logits


In [ ]:
# Train / Validation split
# Call make_train_val_split() at the top of Cell 13 once patient_dirs is known.

def make_train_val_split(patient_dirs, val_fraction=0.2, seed=42):
    """
    Randomly split patient directories into train and val sets.
    Returns (train_dirs, val_dirs, val_pairs).
    """
    rng = random.Random(seed)
    dirs = list(patient_dirs)
    rng.shuffle(dirs)
    split = int(len(dirs) * (1 - val_fraction))
    train_dirs = dirs[:split]
    val_dirs   = dirs[split:]

    val_pairs = []
    for d in val_dirs:
        val_pairs.extend(list_slice_pairs(d))

    print(f"Train patients : {len(train_dirs)}")
    print(f"Val   patients : {len(val_dirs)}")
    print(f"Val   slices   : {len(val_pairs)}")
    return train_dirs, val_dirs, val_pairs

In [ ]:
# =============================================================================
# Sequential Curriculum Training (SCT) for Soft Prompts
# =============================================================================
# Motivation
# ----------
# Standard soft-prompt tuning optimises all T tokens jointly.  Because
# gradient signals are shared, early training can collapse tokens toward a
# common attractor -- a form of stochastic interference that reduces prompt
# diversity.  SCT eliminates this by training tokens in expanding windows:
#
#   Stage 1 : window=1  -> optimise token 0 only until converged
#   Stage 2 : window=2  -> unlock token 1, co-optimise {0,1}
#   Stage 3 : window=4  -> unlock tokens 2-3, co-optimise {0..3}
#   ...  (doubling each stage)
#   Stage k : window=T  -> all T tokens active
#
# Each stage runs until its local early-stopping criterion fires or
# stage_steps is exhausted.  Already-converged tokens stay unfrozen
# so later tokens can build on a stable prefix.
#
# Paper framing:
#   'We introduce Sequential Curriculum Training (SCT), a token-wise
#    curriculum that progressively expands the set of trainable soft-prompt
#    tokens from a single token to the full prompt length.  SCT reduces
#    stochastic interference during early optimisation and produces more
#    interpretable, diverse prompt embeddings than joint training.'
# =============================================================================

def _freeze_tokens(soft_prompt, active_end: int):
    # Zero-gradient hook for tokens beyond the current SCT window.
    # Using a hook (not slicing) preserves optimizer momentum across stages.
    if hasattr(soft_prompt, '_sct_hook') and soft_prompt._sct_hook is not None:
        soft_prompt._sct_hook.remove()
        soft_prompt._sct_hook = None

    def _mask_grad(grad):
        masked = grad.clone()
        masked[active_end:] = 0.0
        return masked

    soft_prompt._sct_hook = soft_prompt.prompt_embeddings.register_hook(_mask_grad)


def train_soft_prompt(
    patient_dirs,
    text_prompt,
    train_prompts=None,
    val_pairs=None,
    epochs=10,
    steps_per_epoch=200,
    batch_size=4,
    lr=2e-4,
    dice_weight=2.5,
    bce_weight=0.5,
    anchor_weight=0.1,
    max_patients=20,
    seed=1337,
    # Per-stage early stopping
    patience=3,
    eval_every=40,
    min_delta=0.001,
    # SCT curriculum (window sizes, defaults to 1->2->4->8->16)
    curriculum_windows=None,
    # Steps budget per curriculum stage
    stage_steps=300,
):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    model.eval()
    for p in model.parameters(): p.requires_grad = False
    soft_prompt.train()
    for p in soft_prompt.parameters(): p.requires_grad = True

    dirs  = patient_dirs[:max_patients] if max_patients else patient_dirs
    pairs = []
    for d in dirs:
        pairs.extend(list_slice_pairs(d))
    if not pairs:
        raise RuntimeError('No (image, mask) slice pairs found.')

    prompt_pool = train_prompts if train_prompts else [text_prompt]

    # Build curriculum window schedule (doublings by default)
    T = soft_prompt.num_tokens
    if curriculum_windows is None:
        curriculum_windows = []
        w = 1
        while w < T:
            curriculum_windows.append(w)
            w *= 2
        curriculum_windows.append(T)
    curriculum_windows = sorted(set(min(w, T) for w in curriculum_windows))
    print(f'[SCT] Curriculum window schedule: {curriculum_windows}')

    # Single optimizer -- momentum states survive across stages
    opt = torch.optim.AdamW(soft_prompt.parameters(), lr=lr, weight_decay=1e-4)

    best_iou_global = 0.0
    best_state      = copy.deepcopy(soft_prompt.state_dict())
    global_step     = 0

    print(f'Training soft prompt on {len(pairs)} slices from {len(dirs)} patients')
    print(f'Prompt pool ({len(prompt_pool)}): {prompt_pool}')
    if val_pairs:
        print(f'Early stopping per stage: patience={patience}, '
              f'eval_every={eval_every}, min_delta={min_delta}')

    for stage_idx, active_end in enumerate(curriculum_windows):
        _freeze_tokens(soft_prompt, active_end)

        print(f'\n{"="*70}')
        print(f'[SCT] Stage {stage_idx+1}/{len(curriculum_windows)} -- '
              f'active tokens: 0..{active_end-1}  ({active_end}/{T})')
        print(f'{"="*70}')

        stage_best_iou   = 0.0
        stage_best_state = copy.deepcopy(soft_prompt.state_dict())
        patience_counter = 0
        stage_stop       = False
        stage_global     = 0

        for step in range(1, stage_steps + 1):
            if stage_stop:
                break

            opt.zero_grad(set_to_none=True)

            seg_loss_sum = torch.tensor(0.0, device=device)
            for _ in range(batch_size):
                img_np, mask_np = load_slice(*random.choice(pairs))
                step_prompt = random.choice(prompt_pool)
                logits = forward_one_slice_logits(img_np, step_prompt)

                gt = torch.from_numpy(mask_np.astype(np.float32)).to(device)
                gt = gt.unsqueeze(0).unsqueeze(0)
                gt = F.interpolate(gt, size=logits.shape[-2:], mode='nearest')

                loss_seg = (bce_weight  * bce_logits(logits, gt)
                          + dice_weight * dice_loss_from_logits(logits, gt))
                seg_loss_sum = seg_loss_sum + loss_seg

            seg_loss_mean = seg_loss_sum / batch_size
            loss_anc  = anchor_manager.anchor_loss(soft_prompt, weight=anchor_weight)
            loss_div  = anchor_manager.diversity_loss(soft_prompt, weight=0.1)
            total_loss = seg_loss_mean + loss_anc + loss_div

            total_loss.backward()
            opt.step()

            stage_global += 1
            global_step  += 1

            if step % 20 == 0:
                print(f'  [Stage {stage_idx+1} | step {step}/{stage_steps}]  '
                      f'total={total_loss.item():.4f}  '
                      f'seg={seg_loss_mean.item():.4f}  '
                      f'anc={loss_anc.item():.4f}  '
                      f'div={loss_div.item():.4f}  '
                      f'active_tokens=0..{active_end-1}')
                print_verbalization(soft_prompt, lang_enc, top_k=1)

            if val_pairs and stage_global % eval_every == 0:
                val_iou  = evaluate_iou(val_pairs, text_prompt)
                improved = val_iou > stage_best_iou + min_delta

                print(f'  [EarlyStopping | Stage {stage_idx+1}]  '
                      f'val_iou={val_iou:.4f}  stage_best={stage_best_iou:.4f}  '
                      f'patience={patience_counter}/{patience}  '
                      f'{"up improved" if improved else "-- no gain"}')

                if improved:
                    stage_best_iou   = val_iou
                    stage_best_state = copy.deepcopy(soft_prompt.state_dict())
                    patience_counter = 0
                    if val_iou > best_iou_global:
                        best_iou_global = val_iou
                        best_state      = copy.deepcopy(soft_prompt.state_dict())
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f'\n  [SCT] Stage {stage_idx+1} converged at step '
                              f'{stage_global}.  Best stage IoU: {stage_best_iou:.4f}')
                        stage_stop = True

        if val_pairs and stage_best_state is not None:
            soft_prompt.load_state_dict(stage_best_state)
            print(f'  [SCT] Stage {stage_idx+1} checkpoint restored '
                  f'(stage IoU={stage_best_iou:.4f})')

        print(f'\n[SCT] Stage {stage_idx+1} verbalization:')
        print_verbalization(soft_prompt, lang_enc, top_k=3)

    # Remove SCT gradient hook
    if hasattr(soft_prompt, '_sct_hook') and soft_prompt._sct_hook is not None:
        soft_prompt._sct_hook.remove()
        soft_prompt._sct_hook = None

    if val_pairs and best_state is not None:
        soft_prompt.load_state_dict(best_state)
        print(f'\n[SCT] Restored globally best soft prompt (val IoU={best_iou_global:.4f})')

    soft_prompt.eval()
    print('Sequential Curriculum Training complete')
    return best_iou_global


In [ ]:
#Configuration

# Paths
OUTPUT_PATH = "/content/drive/MyDrive/SAM_results_biomedparse"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Text prompt for brain tumor
TEXT_PROMPT = "Brain tumor in MRI"

print(f"  Dataset: {DATASET_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Prompt: {TEXT_PROMPT}")

  Dataset: /content/dataset/lgg-mri-segmentation/kaggle_3m
  Output: /content/drive/MyDrive/SAM_results_biomedparse
  Prompt: Brain tumor in MRI


In [ ]:
#Functions
def segment_3d_volume(volume, text_prompt, slice_batch_size=4):
    """
    Segment a 3D volume using BiomedParse

    Args:
        volume: numpy array of shape (D, H, W) - depth, height, width
        text_prompt: text description of what to segment
        slice_batch_size: number of slices to process at once

    Returns:
        segmentation mask of same shape as input
    """
    # Prepare input
    imgs, pad_width, padded_size, valid_axis = process_input(volume, 512)
    imgs = imgs.to(device).int()

    input_tensor = {
        "image": imgs.unsqueeze(0),  # Add batch dimension
        "text": [text_prompt],
    }

    # Run inference — everything under no_grad; no training happens here
    with torch.no_grad():
        if USE_SOFT_PROMPT:
            with patch_get_text_token_embeddings(lang_enc, soft_prompt):
                output = model(input_tensor, mode="eval", slice_batch_size=slice_batch_size)
        else:
            output = model(input_tensor, mode="eval", slice_batch_size=slice_batch_size)

    mask_preds = output["predictions"]["pred_gmasks"]
    mask_preds = F.interpolate(mask_preds, size=(512, 512), mode="bicubic",
                              align_corners=False, antialias=True)

    mask_preds = postprocess(mask_preds, output["predictions"]["object_existence"])

    # Merge to binary mask (we only have one class: tumor)
    mask_preds = merge_multiclass_masks(mask_preds, [1])

    # Process output to original size
    mask_preds = process_output(mask_preds, pad_width, padded_size, valid_axis)

    # Convert to binary (0 or 1)
    mask_preds = (mask_preds > 0.5).astype(np.uint8)

    return mask_preds

def load_patient_volume_from_tif(patient_dir):
    """
    Load all TIF slices into a 3D volume

    Returns:
        volume: numpy array (D, H, W)
        image_files: list of image file paths
    """
    # Get all image files
    image_files = sorted([
        f for f in glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])

    if len(image_files) == 0:
        return None, None

    # Load all slices
    slices = []
    for img_file in image_files:
        img = np.array(Image.open(img_file).convert('L'))
        slices.append(img)

    # Stack into 3D volume (D, H, W)
    volume = np.stack(slices, axis=0)

    return volume, image_files

def process_patient(patient_dir, patient_id, text_prompt):
    """
    Process a single patient
    """
    print(f"  Loading volume...")
    volume, image_files = load_patient_volume_from_tif(patient_dir)

    if volume is None:
        print(f"    ⚠️  No images found")
        return None

    print(f"    Volume shape: {volume.shape}")

    # Segment using BiomedParse
    print(f"  Running BiomedParse segmentation...")
    pred_mask = segment_3d_volume(volume, text_prompt, slice_batch_size=4)

    print(f"    Segmentation shape: {pred_mask.shape}")

    return pred_mask, image_files

In [ ]:
# =============================================================================
# LLM-Guided Hard Prompt Optimization (LGPO) -- FREE COLAB VERSION
# Uses a free Hugging Face instruct model instead of Anthropic Claude
# =============================================================================

# Install dependencies in Colab if needed:
# !pip install -q transformers accelerate sentencepiece

import json
import time
import random
import numpy as np
import torch

from transformers import pipeline
from sklearn.metrics import jaccard_score


# ── System prompt: gives the LLM its role and constraints ────────────────────
LGPO_SYSTEM_PROMPT = """You are a radiologist and medical AI specialist helping to optimize text prompts for a brain tumor segmentation model called BiomedParse.

BiomedParse is a vision-language segmentation model. It takes an MRI image and a text prompt describing what to segment, then produces a binary mask.

Your task is to propose text prompts that maximize Intersection over Union (IoU) between the predicted mask and the ground truth tumor mask on the LGG (Lower Grade Glioma) MRI dataset from TCGA.

Rules for your prompts:
- Be specific and clinically accurate
- Describe the appearance of LGG tumors in MRI
- Keep prompts under 20 words
- Do not repeat a prompt that has already been tried
- Output ONLY the prompt text, nothing else
"""


# ── Load free LLM in Colab ────────────────────────────────────────────────────
# Recommended first choice for Colab:
#   "microsoft/Phi-3-mini-4k-instruct"
# Lighter fallback:
#   "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

FREE_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

generator = pipeline(
    "text-generation",
    model=FREE_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)


def build_lgpo_user_message(history):
    """
    Build the user message sent to the LLM each iteration.
    history: list of {"prompt": str, "iou": float, "iteration": int}
    """
    if not history:
        return (
            "No prompts have been tried yet. Propose your first prompt for "
            "segmenting brain tumors in LGG MRI images."
        )

    sorted_history = sorted(
        [h for h in history if h.get("iou") is not None],
        key=lambda x: x["iou"],
        reverse=True
    )

    if not sorted_history:
        return (
            "No valid prompts have been scored yet. Propose a new prompt for "
            "segmenting brain tumors in LGG MRI images."
        )

    lines = ["Here are all prompts tried so far, ranked by IoU (higher is better):\n"]
    for i, entry in enumerate(sorted_history, 1):
        lines.append(
            f'  {i}. IoU={entry["iou"]:.4f} | iter={entry["iteration"]} '
            f'| prompt: "{entry["prompt"]}"'
        )

    best = sorted_history[0]
    lines.append(f'\nBest so far: "{best["prompt"]}" (IoU={best["iou"]:.4f})')
    lines.append(
        "\nBased on this history, propose ONE new prompt that you predict will "
        "achieve a higher IoU. Output only the prompt text."
    )

    return "\n".join(lines)


def generate_prompt_with_free_llm(user_msg, history, max_retries=3):
    """
    Ask the free Hugging Face model for a new prompt.
    Includes cleanup and duplicate avoidance.
    """
    prior_prompts = {h["prompt"].strip().lower() for h in history if "prompt" in h}

    full_prompt = f"""<|system|>
{LGPO_SYSTEM_PROMPT}
<|user|>
{user_msg}
<|assistant|>
"""

    fallback_prompt = "hyperintense lower grade glioma region on brain flair mri"

    for _ in range(max_retries):
        response = generator(
            full_prompt,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            return_full_text=True
        )

        generated = response[0]["generated_text"]

        # Extract assistant continuation
        if "<|assistant|>" in generated:
            proposed_prompt = generated.split("<|assistant|>")[-1].strip()
        else:
            proposed_prompt = generated.strip()

        # Keep only first line
        proposed_prompt = proposed_prompt.split("\n")[0].strip()

        # Cleanup
        proposed_prompt = proposed_prompt.strip('"').strip("'").strip()
        proposed_prompt = proposed_prompt.rstrip(".!,;:")

        # Enforce short prompt
        words = proposed_prompt.split()
        if len(words) > 20:
            proposed_prompt = " ".join(words[:20])

        # Basic validity checks
        if not proposed_prompt:
            continue

        lowered = proposed_prompt.lower()
        if lowered in prior_prompts:
            continue

        return proposed_prompt

    return fallback_prompt


@torch.no_grad()
def evaluate_hard_prompt_iou(text_prompt, val_pairs, num_samples=80):
    """
    Run BiomedParse inference with soft prompt active and evaluate IoU
    on a sample of validation pairs using the given hard prompt.

    Assumes these already exist in your notebook:
      - soft_prompt
      - load_slice(img_path, mask_path)
      - forward_one_slice_logits(img_np, text_prompt)
    """
    soft_prompt.eval()
    samples = random.sample(val_pairs, min(num_samples, len(val_pairs)))
    ious = []

    for img_path, mask_path in samples:
        img_np, mask_np = load_slice(img_path, mask_path)
        logits = forward_one_slice_logits(img_np, text_prompt)
        pred = (torch.sigmoid(logits) > 0.5).cpu().numpy().squeeze()

        if pred.shape != mask_np.shape:
            from scipy.ndimage import zoom
            scale = [
                mask_np.shape[0] / pred.shape[0],
                mask_np.shape[1] / pred.shape[1]
            ]
            pred = zoom(pred.astype(np.float32), scale, order=0).astype(np.uint8)

        gt_sum, pred_sum = mask_np.sum(), pred.sum()

        if gt_sum == 0 and pred_sum == 0:
            ious.append(1.0)
        elif gt_sum == 0 or pred_sum == 0:
            ious.append(0.0)
        else:
            ious.append(
                jaccard_score(
                    mask_np.flatten(),
                    pred.flatten(),
                    zero_division=0
                )
            )

    return float(np.mean(ious)) if ious else 0.0


def run_lgpo(
    val_pairs,
    n_iterations=20,
    num_eval_samples=80,
    save_path=None,
):
    """
    LLM-Guided Prompt Optimization loop using a free Hugging Face model.

    Parameters
    ----------
    val_pairs         : list of (img_path, mask_path) tuples
    n_iterations      : number of LLM proposal + inference cycles
    num_eval_samples  : val slices sampled per evaluation
    save_path         : if set, saves full history as JSON after every iteration
    """
    history = []
    best_prompt = None
    best_iou = 0.0

    print(f"\n{'=' * 70}")
    print("[LGPO] Starting LLM-Guided Prompt Optimization")
    print(f"[LGPO] Model: {FREE_MODEL_NAME}")
    print(f"[LGPO] Iterations: {n_iterations} | Eval samples: {num_eval_samples}")
    print(f"{'=' * 70}\n")

    for iteration in range(1, n_iterations + 1):
        print(f"[LGPO] Iteration {iteration}/{n_iterations}")

        # Step 1: Ask free LLM for next prompt
        user_msg = build_lgpo_user_message(history)
        proposed_prompt = generate_prompt_with_free_llm(user_msg, history)
        print(f'  LLM proposed: "{proposed_prompt}"')

        prior_prompts = [h["prompt"] for h in history]
        if proposed_prompt in prior_prompts:
            print(f"  [LGPO] Duplicate prompt — skipping iteration {iteration}")
            history.append({
                "iteration": iteration,
                "prompt": proposed_prompt,
                "iou": None,
                "note": "duplicate"
            })
            continue

        # Step 2: Evaluate
        print(f"  Running inference on {num_eval_samples} val slices...")
        iou = evaluate_hard_prompt_iou(
            proposed_prompt,
            val_pairs,
            num_samples=num_eval_samples
        )
        print(f"  IoU = {iou:.4f}")

        # Step 3: Record result
        entry = {
            "iteration": iteration,
            "prompt": proposed_prompt,
            "iou": iou,
        }
        history.append(entry)

        if iou > best_iou:
            best_iou = iou
            best_prompt = proposed_prompt
            print(f'  *** New best: "{best_prompt}" (IoU={best_iou:.4f}) ***')

        # Step 4: Save checkpoint
        if save_path:
            with open(save_path, "w") as f:
                json.dump(
                    {
                        "best_prompt": best_prompt,
                        "best_iou": best_iou,
                        "history": history,
                    },
                    f,
                    indent=2
                )

        print(f'  Running best so far: "{best_prompt}" | IoU={best_iou:.4f}\n')
        time.sleep(0.5)

    print(f"\n{'=' * 70}")
    print("[LGPO] Optimization complete")
    print(f'[LGPO] Best prompt : "{best_prompt}"')
    print(f"[LGPO] Best IoU    : {best_iou:.4f}")
    print(f"{'=' * 70}")

    valid_history = [h for h in history if h.get("iou") is not None]
    ious_over_time = [h["iou"] for h in valid_history]
    print("\n[LGPO] IoU trajectory: " + " → ".join(f"{x:.3f}" for x in ious_over_time))

    return best_prompt, best_iou, history

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
#Process Patients

# Load patient info
data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)
patient_dirs = sorted(glob(os.path.join(DATASET_PATH, "TCGA_*")))

print(f"Patients in dataset: {len(patient_info)}")
print(f"Patient directories found: {len(patient_dirs)}")

# Split into train / val (80/20) — val is used for early stopping only
train_dirs, val_dirs, val_pairs = make_train_val_split(patient_dirs, val_fraction=0.2)

results = []




# ============================
# 🔥 ADD LLM HERE
# ============================

best_prompt, best_prompt_iou, history = run_lgpo(
    val_pairs=val_pairs,
    n_iterations=20,
    num_eval_samples=80,
    save_path="/content/lgpo_history.json"
)

print(f"\n[LGPO] Best hard prompt: {best_prompt}")
print(f"[LGPO] Best IoU: {best_prompt_iou:.4f}")

# Replace original prompt with optimized one
TEXT_PROMPT = best_prompt

# Print final verbalization of the trained soft prompt
print_verbalization(soft_prompt, lang_enc, top_k=3)


for idx, patient_dir in enumerate(patient_dirs, 1):
    full_patient_id  = os.path.basename(patient_dir)
    short_patient_id = '_'.join(full_patient_id.split('_')[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {short_patient_id}")

    # Get demographics
    patient_row = patient_info[patient_info['Patient'] == short_patient_id]
    if len(patient_row) == 0:
        print("  No demographics, skipping")
        continue

    race   = patient_row.iloc[0]['race']
    gender = patient_row.iloc[0]['gender']
    print(f"  Race: {race}, Gender: {gender}")

    try:
        result = process_patient(patient_dir, short_patient_id, TEXT_PROMPT)

        if result is None:
            continue

        pred_volume, image_files = result
        print(f"  Segmentation complete")

        # Save NIfTI
        affine   = np.eye(4)
        out_path = os.path.join(OUTPUT_PATH, f"{short_patient_id}_biomedparse_predictions.nii.gz")
        nib.save(nib.Nifti1Image(pred_volume, affine), out_path)
        print(f"  Saved: {out_path}")

        # Calculate metrics against ground truth
        gt_pairs = []
        for img_file in image_files:
            base      = os.path.splitext(img_file)[0]
            mask_file = f"{base}_mask.tif"
            if os.path.exists(mask_file):
                gt_pairs.append((img_file, mask_file))

        ious = []
        f1s  = []

        for z, (_, mask_file) in enumerate(gt_pairs):
            if z >= pred_volume.shape[0]:
                break

            gt_slice   = np.array(Image.open(mask_file).convert("L"))
            gt_binary  = (gt_slice > 0).astype(np.uint8)
            pred_slice = pred_volume[z]

            if gt_binary.shape != pred_slice.shape:
                from scipy.ndimage import zoom
                scale_factors = [gt_binary.shape[0] / pred_slice.shape[0],
                                 gt_binary.shape[1] / pred_slice.shape[1]]
                pred_slice = zoom(pred_slice, scale_factors, order=0).astype(np.uint8)

            gt_sum   = gt_binary.sum()
            pred_sum = pred_slice.sum()

            if gt_sum == 0 and pred_sum == 0:
                ious.append(1.0); f1s.append(1.0)
            elif gt_sum == 0 and pred_sum > 0:
                ious.append(0.0); f1s.append(0.0)
            elif gt_sum > 0 and pred_sum == 0:
                ious.append(0.0); f1s.append(0.0)
            else:
                ious.append(jaccard_score(gt_binary.flatten(), pred_slice.flatten(), zero_division=0))
                f1s.append(f1_score(gt_binary.flatten(), pred_slice.flatten(), zero_division=0))

        mean_iou = np.mean(ious) if ious else 0
        mean_f1  = np.mean(f1s)  if f1s  else 0
        print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f}")

        results.append({
            'Patient':         short_patient_id,
            'Race':            race,
            'Gender':          gender,
            'IoU':             mean_iou,
            'F1':              mean_f1,
            'Num_Slices':      len(ious),
            'Prediction_Path': out_path
        })

    except Exception as e:
        print(f"   Error: {str(e)[:200]}")
        import traceback
        traceback.print_exc()
        continue

Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Patients in dataset: 110
Patient directories found: 110
Train patients : 88
Val   patients : 22
Val   slices   : 796

[LGPO] Starting LLM-Guided Prompt Optimization
[LGPO] Model: google/gemma-2-2b-it
[LGPO] Iterations: 20 | Eval samples: 80

[LGPO] Iteration 1/20
  LLM proposed: "LGG tumor, characterized by hyperintense areas on T2-weighted image"
  Running inference on 80 val slices...
  IoU = 0.2671
  *** New best: "LGG tumor, characterized by hyperintense areas on T2-weighted image" (IoU=0.2671) ***
  Running best so far: "LGG tumor, characterized by hyperintense areas on T2-weighted image" | IoU=0.2671



Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LGPO] Iteration 2/20
  LLM proposed: "LGG tumor, enhancing in T1-weighted image"
  Running inference on 80 val slices...
  IoU = 0.3344
  *** New best: "LGG tumor, enhancing in T1-weighted image" (IoU=0.3344) ***
  Running best so far: "LGG tumor, enhancing in T1-weighted image" | IoU=0.3344



Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  LLM proposed: "hyperintense lower grade glioma region on brain flair mri"
  [LGPO] Duplicate prompt — skipping iteration 18
[LGPO] Iteration 19/20


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  LLM proposed: "hyperintense lower grade glioma region on brain flair mri"
  [LGPO] Duplicate prompt — skipping iteration 19
[LGPO] Iteration 20/20


Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  LLM proposed: "hyperintense lower grade glioma region on brain flair mri"
  [LGPO] Duplicate prompt — skipping iteration 20

[LGPO] Optimization complete
[LGPO] Best prompt : "LGG tumor, hyperintense on T2-weighted image, enhancing in T1-weighted image"
[LGPO] Best IoU    : 0.3783

[LGPO] IoU trajectory: 0.267 → 0.334 → 0.264 → 0.249 → 0.378 → 0.330 → 0.354 → 0.378 → 0.343 → 0.312 → 0.312 → 0.351 → 0.340 → 0.360 → 0.367

[LGPO] Best hard prompt: LGG tumor, hyperintense on T2-weighted image, enhancing in T1-weighted image
[LGPO] Best IoU: 0.3783

[SSMV] Slot-Structured Medical Verbalization:
  Token    Slot           Top matches
  -----------------------------------------------------------------
  Token  0  [lesion      ]  "meningioma" (0.071), "tumor" (0.061), "glioma" (0.056)
  Token  1  [anatomy     ]  "periventricular" (0.096), "parietal" (0.060), "frontal" (0.046)
  Token  2  [modality    ]  "contrast-enhanced" (0.053), "post-contrast" (0.028), "axial" (0.024)
  Token  3  [morpho

In [ ]:
#Save Results

if len(results) > 0:
    results_df = pd.DataFrame(results)

    # Save detailed results
    csv_path = os.path.join(OUTPUT_PATH, "biomedparse_patient_metrics.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")

    # Overall metrics
    overall_iou = results_df['IoU'].mean()
    overall_f1 = results_df['F1'].mean()

    print(f"\n{'='*70}")
    print("OVERALL METRICS")
    print(f"{'='*70}")
    print(f"  Mean IoU: {overall_iou:.4f}")
    print(f"  Mean F1:  {overall_f1:.4f}")
    print(f"  Patients: {len(results_df)}")

    # Race-level metrics
    race_stats = results_df.groupby('Race')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("RACE-LEVEL METRICS")
    print(f"{'='*70}")
    print(race_stats)
    race_stats.to_csv(os.path.join(OUTPUT_PATH, "biomedparse_race_metrics.csv"))

    # Gender-level metrics
    gender_stats = results_df.groupby('Gender')[['IoU', 'F1']].agg(['mean', 'std', 'count']).round(4)
    print(f"\n{'='*70}")
    print("GENDER-LEVEL METRICS")
    print(f"{'='*70}")
    print(gender_stats)
    gender_stats.to_csv(os.path.join(OUTPUT_PATH, "biomedparse_gender_metrics.csv"))

    # Save summary
    summary_path = os.path.join(OUTPUT_PATH, "biomedparse_summary.txt")
    with open(summary_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("BiomedParse SEGMENTATION RESULTS\n")
        f.write("="*70 + "\n\n")

        f.write("OVERALL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"Mean IoU: {overall_iou:.4f}\n")
        f.write(f"Mean F1:  {overall_f1:.4f}\n")
        f.write(f"Patients: {len(results_df)}\n\n")

        f.write("RACE-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(race_stats.to_string())
        f.write("\n\n")

        f.write("GENDER-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(gender_stats.to_string())
        f.write("\n")

    print(f" Saved: {summary_path}")

    print("\n" + "="*70)
    print("🎉 BATCH PROCESSING COMPLETE!")
    print("="*70)
    print(f"\nResults saved to: {OUTPUT_PATH}")
    print("\nFiles created:")
    print("  - biomedparse_patient_metrics.csv")
    print("  - biomedparse_race_metrics.csv")
    print("  - biomedparse_gender_metrics.csv")
    print("  - biomedparse_summary.txt")
    print("  - {patient}_biomedparse_predictions.nii.gz (for each patient)")
else:
    print(" No results to save")

Saved: /content/drive/MyDrive/SAM_results_biomedparse/biomedparse_patient_metrics.csv

OVERALL METRICS
  Mean IoU: 0.6577
  Mean F1:  0.6741
  Patients: 110

RACE-LEVEL METRICS
         IoU                    F1              
        mean     std count    mean     std count
Race                                            
2.0   0.6525  0.1392    10  0.6666  0.1412    10
3.0   0.6549  0.1173    98  0.6715  0.1166    98

GENDER-LEVEL METRICS
           IoU                    F1              
          mean     std count    mean     std count
Gender                                            
1.0     0.6717  0.1114    56  0.6889  0.1114    56
2.0     0.6393  0.1254    53  0.6546  0.1239    53
 Saved: /content/drive/MyDrive/SAM_results_biomedparse/biomedparse_summary.txt

🎉 BATCH PROCESSING COMPLETE!

Results saved to: /content/drive/MyDrive/SAM_results_biomedparse

Files created:
  - biomedparse_patient_metrics.csv
  - biomedparse_race_metrics.csv
  - biomedparse_gender_metrics.csv
  - bi